# 00 — Patent Search & Filter
Side quest: explore and filter the Excel dataset before running the pipeline.
Use this notebook to find the 90 patents you want to work with.

In [ ]:
import sys; sys.path.insert(0, "..")
import pandas as pd
import yaml
from src.patents import load_patents

with open("../config.yaml") as f:
    cfg = yaml.safe_load(f)

df = load_patents(cfg)
print(f"Total patents: {len(df)}")
df.head(3)

## Explore columns

In [ ]:
# What industries / domains are in the dataset?
print("Record Types:")
print(df["Record Type"].value_counts())
print()
print("Legal Status:")
print(df["Family Legal Status(Dead/Alive)"].value_counts())
print()
print("Tech Sub Domain (top 10):")
print(df["Tech Sub Domain"].value_counts().head(10))

## Filter to your target subset
Edit filters below and run to see how many patents match.

In [ ]:
# ── Edit these filters ──────────────────────────
RECORD_TYPE   = "Patent"        # or None
LEGAL_STATUS  = "ALIVE"         # or None
DOMAIN_CONTAINS = "TRANSPORT"   # partial match, or None
DATE_FROM     = "2020-01-01"    # or None
# ────────────────────────────────────────────────

filtered = df.copy()
if RECORD_TYPE:
    filtered = filtered[filtered["Record Type"] == RECORD_TYPE]
if LEGAL_STATUS:
    filtered = filtered[filtered["Family Legal Status(Dead/Alive)"] == LEGAL_STATUS]
if DOMAIN_CONTAINS:
    filtered = filtered[filtered["Tech Sub Domain"].str.contains(DOMAIN_CONTAINS, na=False)]
if DATE_FROM:
    filtered = filtered[pd.to_datetime(filtered["Publication/Issue Date"], errors="coerce") >= DATE_FROM]

print(f"Filtered: {len(filtered)} patents")
filtered[["Record Number", "Title", "Record Type", "Family Legal Status(Dead/Alive)", "Publication/Issue Date"]].head(10)

## Save your filtered list
Once happy with the filter, save the Record Numbers so Phase 1 can use them.

In [ ]:
# Save filtered Record Numbers to a text file for use in other notebooks
out = "../selected_patents.txt"
filtered["Record Number"].to_csv(out, index=False, header=False)
print(f"Saved {len(filtered)} patent IDs to {out}")